# Normalize Kobzar and train BPE tokenizer

This notebook keeps spaces, newlines, and punctuation, lowercases/NFKC-normalizes the text, then trains a Hugging Face BPE tokenizer. Punctuation is isolated during pre-tokenization, so BPE cannot learn merged tokens like `слово.`.

In [ ]:
import re
import unicodedata
from pathlib import Path

DATA_DIR = Path.cwd()
INPUT_PATH = DATA_DIR / "kobzar.txt"
OUTPUT_DIR = DATA_DIR / "kobzar-bpe"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "kobzar.normalized.txt"

UNKNOWN = "<|unknown|>"
END_OF_TEXT = "<|endoftext|>"
END_OF_PART = "<|endofpart|>"
SPECIAL_TOKENS = [UNKNOWN, END_OF_TEXT, END_OF_PART]
UKRAINIAN_LETTERS = set("абвгґдеєжзиіїйклмнопрстуфхцчшщьюя")
PUNCTUATION = set(".,!?;:…'\"«»“”„’`´()-—–-[]{}<>/\\")
ALLOWED_CHARS = UKRAINIAN_LETTERS | PUNCTUATION | {" ", "\n"}
TRANSLATION_TABLE = str.maketrans({
    "\r": "\n",
    "\t": " ",
    "\f": " ",
    "\v": " ",
    "ʼ": "'",
    "’": "'",
    "`": "'",
    "´": "'",
    "«": '"',
    "»": '"',
    "“": '"',
    "”": '"',
    "„": '"',
    "—": "-",
    "–": "-",
    "−": "-",
    "…": "...",
})
SEPARATOR_REPLACEMENTS = [
    (re.compile(r"(?m)^\s*-{3,}\s*$"), END_OF_TEXT),
    (re.compile(r"(?m)^\s*(?:\*\s*){3,}\s*$"), END_OF_PART),
]


def protect_special_tokens(text: str) -> tuple[str, dict[str, str]]:
    placeholders = {"\uE000": END_OF_TEXT, "\uE001": END_OF_PART}
    for placeholder, token in placeholders.items():
        text = text.replace(token, placeholder)
    return text, placeholders


def restore_special_placeholders(text: str, placeholders: dict[str, str]) -> str:
    for placeholder, token in placeholders.items():
        text = text.replace(placeholder, token)
    return text


def replace_separators(text: str) -> str:
    for pattern, token in SEPARATOR_REPLACEMENTS:
        text = pattern.sub(token, text)
    return text


def normalize_kobzar_text(text: str) -> str:
    text = replace_separators(text)
    text, placeholders = protect_special_tokens(text)
    text = unicodedata.normalize("NFKC", text).lower().translate(TRANSLATION_TABLE)
    placeholder_chars = set(placeholders)

    chars = []
    for ch in text:
        if ch in placeholder_chars:
            chars.append(ch)
        elif ch in UKRAINIAN_LETTERS or ch in PUNCTUATION or ch == "\n":
            chars.append(ch)
        elif ch.isspace():
            chars.append(" ")
        else:
            chars.append(" ")

    normalized = "".join(chars)
    normalized = re.sub(r"[^\S\n]+", " ", normalized)
    normalized = re.sub(r" *\n *", "\n", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    normalized = restore_special_placeholders(normalized, placeholders)
    normalized = re.sub(rf" *({re.escape(END_OF_TEXT)}|{re.escape(END_OF_PART)}) *", r"\1", normalized)
    return normalized.strip()

raw_text = INPUT_PATH.read_text(encoding="utf-8")
normalized = normalize_kobzar_text(raw_text)
OUTPUT_PATH.write_text(normalized, encoding="utf-8")

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Original characters: {len(raw_text)}")
print(f"Normalized characters: {len(normalized)}")
print(f"Words: {len(normalized.split())}")
print(f"Unique normalized characters: {''.join(sorted(set(normalized)))}")
print(f"Preview: {normalized[:500]}")

In [ ]:
# Smoke check: lowercase Ukrainian text, punctuation, spaces, and newlines only.
unexpected_chars = sorted(set(normalized) - ALLOWED_CHARS)
assert not unexpected_chars, unexpected_chars
assert normalized == normalized.lower()
assert "  " not in normalized
assert " \n" not in normalized
assert "\n " not in normalized

print("OK: normalized text keeps spaces/newlines/punctuation and removes unsupported characters.")

## Train BPE tokenizer

The actual model input is `ids.txt`. `tokens.debug.txt` is only a human-readable debug dump. The tokenizer itself is saved as `tokenizer.json`.

In [ ]:
# Run this once if the tokenizers package is not installed in the current environment.
try:
    import tokenizers
    print(f"tokenizers is already installed: {tokenizers.__version__}")
except ModuleNotFoundError:
    %pip install tokenizers

In [ ]:
from tokenizers import Regex, Tokenizer, decoders, normalizers, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

TOKENIZER_PATH = OUTPUT_DIR / "tokenizer.json"
IDS_PATH = OUTPUT_DIR / "ids.txt"
TOKENS_DEBUG_PATH = OUTPUT_DIR / "tokens.debug.txt"

def split_on_next_newline(text: str, target_size: int = 2048) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        target = start + target_size
        if target >= len(text):
            chunks.append(text[start:])
            break

        end = text.find("
", target)
        if end == -1:
            chunks.append(text[start:])
            break

        end += 1  # keep the newline at the end of the chunk
        chunks.append(text[start:end])
        start = end

    return chunks

chunk_size = 2048
chunks = split_on_next_newline(normalized, chunk_size)

tokenizer = Tokenizer(BPE(unk_token=UNKNOWN))
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFKC(),
    normalizers.Lowercase(),
])
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(r"<\|(?:unknown|endoftext|endofpart)\|>"), "isolated"),
    # Preserve spaces as word-boundary markers, GPT-style, but do not add a fake prefix space.
    pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="never"),
    # Newlines stay visible and isolated.
    pre_tokenizers.Split(Regex(r"\n+"), "isolated"),
    # Punctuation is isolated before BPE merges, so BPE cannot learn tokens like "слово.".
    pre_tokenizers.Punctuation("isolated"),
])
tokenizer.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="never")

trainer = BpeTrainer(
    vocab_size=2049,
    min_frequency=2,
    special_tokens=[UNKNOWN],
    initial_alphabet=sorted(UKRAINIAN_LETTERS | PUNCTUATION | {"▁", "\n"}),
)

training_chunks = [chunk.replace(END_OF_TEXT, "\n").replace(END_OF_PART, "\n") for chunk in chunks]
if training_chunks:
    training_chunks[0] = " " + training_chunks[0]
tokenizer.train_from_iterator(training_chunks, trainer=trainer, length=len(training_chunks))

# Add special tokens after training. UNKNOWN is the zero token; EOT/EOP follow the BPE vocab.
added_special_tokens = tokenizer.add_special_tokens([END_OF_TEXT, END_OF_PART])
assert added_special_tokens == 2
assert tokenizer.token_to_id(UNKNOWN) == 0
assert tokenizer.token_to_id(END_OF_TEXT) == 2049
assert tokenizer.token_to_id(END_OF_PART) == 2050

tokenizer.save(str(TOKENIZER_PATH))

all_ids = []
all_tokens = []
for chunk in chunks:
    encoded_chunk = tokenizer.encode(chunk)
    all_ids.extend(encoded_chunk.ids)
    all_tokens.extend(encoded_chunk.tokens)

IDS_PATH.write_text(" ".join(map(str, all_ids)), encoding="utf-8")
TOKENS_DEBUG_PATH.write_text("\n".join(all_tokens), encoding="utf-8")

print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Token IDs: {IDS_PATH}")
print(f"Debug tokens: {TOKENS_DEBUG_PATH}")
print(f"Vocab size: {tokenizer.get_vocab_size()}")
print(f"Chunks: {len(chunks)}")
print(f"Encoded token count: {len(all_ids)}")
print(f"First 50 debug tokens: {all_tokens[:50]}")
print(f"First 50 ids: {all_ids[:50]}")

In [ ]:
# Smoke check for BPE artifacts and punctuation isolation.
decoded = tokenizer.decode(all_ids, skip_special_tokens=False)
assert decoded == normalized
assert END_OF_TEXT in normalized
assert END_OF_PART in normalized
assert tokenizer.token_to_id(END_OF_TEXT) is not None
assert tokenizer.token_to_id(END_OF_PART) is not None
assert tokenizer.token_to_id(END_OF_TEXT) == 2048
assert tokenizer.token_to_id(END_OF_PART) == 2049

punctuation_after_translation = set(ch.translate(TRANSLATION_TABLE) for ch in PUNCTUATION)
punctuation_after_translation = set("".join(punctuation_after_translation))
bad_word_punctuation_tokens = [
    token for token in all_tokens
    if any(ch in UKRAINIAN_LETTERS for ch in token)
    and any(ch in punctuation_after_translation for ch in token)
]
assert not bad_word_punctuation_tokens, bad_word_punctuation_tokens[:20]

assert TOKENIZER_PATH.exists()
assert IDS_PATH.exists()
assert TOKENS_DEBUG_PATH.exists()
assert len(all_ids) > 0

normalized_char_count = len(normalized)
token_count = len(all_ids)
compression_rate = normalized_char_count / token_count if token_count else 0

print("OK: BPE artifacts exist, decode is exact, and punctuation is not merged with word tokens.")
print(f"Normalized chars: {normalized_char_count}")
print(f"Token count: {token_count}")
print(f"Compression rate: {compression_rate:.4f}")

## Vocabulary size compression benchmark

Train temporary BPE tokenizers for several vocabulary sizes and compare token count / compression rate on the normalized text. Nothing is saved from this benchmark.

In [ ]:
import time

benchmark_vocab_sizes = list(range(512, 4096 + 1, 512))
benchmark_results = []

for benchmark_vocab_size in benchmark_vocab_sizes:
    started_at = time.perf_counter()

    benchmark_tokenizer = Tokenizer(BPE(unk_token=UNKNOWN))
    benchmark_tokenizer.normalizer = normalizers.Sequence([
        normalizers.NFKC(),
        normalizers.Lowercase(),
    ])
    benchmark_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(r"<\|(?:unknown|endoftext|endofpart)\|>"), "isolated"),
        pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="never"),
        pre_tokenizers.Split(Regex(r"
+"), "isolated"),
        pre_tokenizers.Punctuation("isolated"),
    ])
    benchmark_tokenizer.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="never")

    benchmark_trainer = BpeTrainer(
        vocab_size=benchmark_vocab_size,
        min_frequency=2,
        special_tokens=[UNKNOWN],
        initial_alphabet=sorted(UKRAINIAN_LETTERS | PUNCTUATION | {"▁", "
"}),
    )
    benchmark_training_chunks = chunks.copy()
    if benchmark_training_chunks:
        benchmark_training_chunks[0] = " " + benchmark_training_chunks[0]
    benchmark_tokenizer.train_from_iterator(benchmark_training_chunks, trainer=benchmark_trainer, length=len(benchmark_training_chunks))

    benchmark_token_count = 0
    for chunk in chunks:
        benchmark_token_count += len(benchmark_tokenizer.encode(chunk).ids)

    elapsed_seconds = time.perf_counter() - started_at
    compression_rate = len(normalized) / benchmark_token_count if benchmark_token_count else 0
    benchmark_results.append((
        benchmark_vocab_size,
        benchmark_tokenizer.get_vocab_size(),
        benchmark_token_count,
        compression_rate,
        elapsed_seconds,
    ))

print(f"{'target_vocab':>12} {'actual_vocab':>12} {'tokens':>12} {'norm_chars/token':>16} {'seconds':>10}")
print("-" * 70)
for target_vocab, actual_vocab, token_count, compression_rate, elapsed_seconds in benchmark_results:
    print(f"{target_vocab:>12} {actual_vocab:>12} {token_count:>12} {compression_rate:>16.4f} {elapsed_seconds:>10.2f}")